# Where Does the Time Go? Step-Level Duration Gap Analysis

**Course:** From Analytics to Action (42577) — Spring 2026  
**Data:** Completed operations from Rigshospitalet, 2024

This notebook decomposes the gap between **planned and actual** OR timelines into discrete steps for three high-volume procedures. The goal is to identify *where* in the process time is lost — is it in getting the room and patient ready (prep → patient in OR), or during the procedure itself (patient in OR → patient leaves OR)?

**Procedures investigated:**
- EKSPLORATIV LAPAROTOMI (exploratory laparotomy — major abdominal surgery)
- JJ-KATETER ANLÆGGELSE (JJ catheter placement — urology)
- GASTROSKOPI (gastroscopy — endoscopy)

---

## Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# Colors
RH_BLUE = '#0077B6'
ACCENT_RED = '#E63946'
ACCENT_GREEN = '#2A9D8F'
ACCENT_AMBER = '#E9C46A'
GREY = '#8D99AE'

PROC_COLORS = {
    'EKSPLORATIV LAPAROTOMI': ACCENT_RED,
    'JJ-KATETER ANLÆGGELSE': RH_BLUE,
    'GASTROSKOPI': ACCENT_GREEN,
}

print('Setup complete.')

In [ ]:
# ============================================================
# LOAD DATA — update the path to your CSV file
# ============================================================
DATA_PATH = 'completed_operations.csv'  # <-- change this

df = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8')
print(f'Loaded {len(df):,} rows × {len(df.columns)} columns')

## Data Preparation

In [ ]:
# --- Parse datetime columns ---
time_cols = [
    'Planlagt stue klargøring start',
    'Stue klargøring start',
    'Patient på stuen (Planlagt)',
    'Patient på stuen',
    'Patient forlader stuen (Planlagt)',
    'Patient forlader stuen',
    'Procedure start',
    'Procedure slut',
    'Dato',
]
for col in time_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')

# --- Extract procedure name ---
df['Procedure_Name'] = (
    df['Procedure - Tekst & ID']
    .str.replace(r'\s*\[.*\]', '', regex=True)
    .str.strip()
)

df['Is_Acute'] = df['Akut case (J/N)'].map({'Ja': True, 'Nej': False})
df['Day_of_Week'] = df['Dato'].dt.day_name()

In [ ]:
# --- Compute step durations (actual and planned, in minutes) ---

def minutes_between(df, col_start, col_end):
    """Return duration in minutes between two datetime columns."""
    return (df[col_end] - df[col_start]).dt.total_seconds() / 60

# Step 1: Prep start → Patient in OR
df['actual_prep_to_inOR'] = minutes_between(df, 'Stue klargøring start', 'Patient på stuen')
df['planned_prep_to_inOR'] = minutes_between(df, 'Planlagt stue klargøring start', 'Patient på stuen (Planlagt)')
df['gap_prep_to_inOR'] = df['actual_prep_to_inOR'] - df['planned_prep_to_inOR']

# Step 2: Patient in OR → Patient leaves OR
df['actual_inOR_to_leave'] = minutes_between(df, 'Patient på stuen', 'Patient forlader stuen')
df['planned_inOR_to_leave'] = minutes_between(df, 'Patient på stuen (Planlagt)', 'Patient forlader stuen (Planlagt)')
df['gap_inOR_to_leave'] = df['actual_inOR_to_leave'] - df['planned_inOR_to_leave']

# Total: Prep start → Patient leaves OR
df['actual_total'] = minutes_between(df, 'Stue klargøring start', 'Patient forlader stuen')
df['planned_total'] = minutes_between(df, 'Planlagt stue klargøring start', 'Patient forlader stuen (Planlagt)')
df['gap_total'] = df['actual_total'] - df['planned_total']

print('Step duration columns created.')

In [ ]:
# --- Filter to the three procedures of interest ---
TARGET_PROCS = ['EKSPLORATIV LAPAROTOMI', 'JJ-KATETER ANLÆGGELSE', 'GASTROSKOPI']

mask = df['Procedure_Name'].isin(TARGET_PROCS)
dff = df[mask].copy()

print(f'Filtered to {len(dff):,} rows across {dff["Case-ID Anonymous"].nunique()} unique cases')
print()
for proc in TARGET_PROCS:
    sub = dff[dff['Procedure_Name'] == proc]
    print(f'  {proc}: {len(sub):,} rows, {sub["Case-ID Anonymous"].nunique()} cases')

---

## Step Duration Gap Summary

For each procedure, we decompose the total time overrun into two steps:

1. **Prep start → Patient in OR** — Room preparation and getting the patient positioned. Overruns here = logistics problem (equipment staging, turnover, transport).
2. **Patient in OR → Patient leaves OR** — The actual procedure plus post-op in-room time. Overruns here = clinical complexity or recovery variability.

A positive gap means the step took longer than planned. A negative gap means it was faster.

In [ ]:
# --- Build the gap summary table ---
gap_cols = {
    'Prep start → Patient in OR': 'gap_prep_to_inOR',
    'Patient in OR → Patient leaves OR': 'gap_inOR_to_leave',
    'Total: Prep start → Patient leaves OR': 'gap_total',
}

rows = []
for proc in TARGET_PROCS:
    sub = dff[dff['Procedure_Name'] == proc]
    for step_label, col in gap_cols.items():
        vals = sub[col].dropna()
        rows.append({
            'Procedure': proc,
            'Step': step_label,
            'n': len(vals),
            'Mean Gap (min)': vals.mean(),
            'Median Gap (min)': vals.median(),
            'P90 Gap (min)': vals.quantile(0.9),
            'Std (min)': vals.std(),
        })

gap_summary = pd.DataFrame(rows)
gap_summary = gap_summary.round(1)
gap_summary

### Key Reading of the Table

The critical pattern: for all three procedures, the **prep-to-patient-in-OR step** accounts for the bulk of the overrun. The in-OR step is comparatively close to plan (or even slightly negative for laparotomy). This tells us the delay is accumulating *before the surgeon even starts working*.

---

## Visualization 1: Where Does the Overrun Accumulate?

Stacked bar chart showing mean gap by step for each procedure.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

step1_col = 'gap_prep_to_inOR'
step2_col = 'gap_inOR_to_leave'

means = []
for proc in TARGET_PROCS:
    sub = dff[dff['Procedure_Name'] == proc]
    means.append({
        'Procedure': proc.title(),
        'Prep → Patient in OR': sub[step1_col].mean(),
        'In OR → Patient Leaves': sub[step2_col].mean(),
    })

means_df = pd.DataFrame(means).set_index('Procedure')

means_df.plot(
    kind='barh', stacked=True, ax=ax,
    color=[ACCENT_AMBER, GREY],
    edgecolor='white', linewidth=0.8
)

# Add total annotation
for i, (proc, row) in enumerate(means_df.iterrows()):
    total = row.sum()
    ax.text(total + 1, i, f'{total:+.0f} min total',
            va='center', fontsize=10, fontweight='bold', color='#333')

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean Gap: Actual − Planned (minutes)')
ax.set_ylabel('')
ax.set_title('Where Does the Time Overrun Accumulate?', fontweight='bold')
ax.legend(title='Step', loc='lower right')
plt.tight_layout()
plt.savefig('step_gap_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Visualization 2: Distribution of Gaps by Step

The means tell the story, but the distributions reveal whether the overruns are consistent or driven by a long tail of extreme cases.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, proc in enumerate(TARGET_PROCS):
    ax = axes[i]
    sub = dff[dff['Procedure_Name'] == proc]
    
    data_step1 = sub['gap_prep_to_inOR'].dropna()
    data_step2 = sub['gap_inOR_to_leave'].dropna()
    
    # Clip extreme outliers for readability
    clip_lo, clip_hi = -60, 180
    
    ax.hist(data_step1.clip(clip_lo, clip_hi), bins=40, alpha=0.6,
            color=ACCENT_AMBER, label='Prep → In OR', edgecolor='white')
    ax.hist(data_step2.clip(clip_lo, clip_hi), bins=40, alpha=0.5,
            color=GREY, label='In OR → Leaves', edgecolor='white')
    
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    ax.axvline(data_step1.mean(), color=ACCENT_AMBER, linewidth=2, linestyle='-',
              label=f'Prep mean: {data_step1.mean():+.0f}')
    ax.axvline(data_step2.mean(), color=GREY, linewidth=2, linestyle='-',
              label=f'In-OR mean: {data_step2.mean():+.0f}')
    
    ax.set_title(proc.title(), fontweight='bold', fontsize=12)
    ax.set_xlabel('Gap (actual − planned, min)')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Count')
fig.suptitle('Distribution of Step Duration Gaps', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('step_gap_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Visualization 3: The P90 Problem

The mean gap might be 30 minutes, but the 90th percentile shows the worst-case reality that schedulers have to plan around. This is the "tail risk" view.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Reshape for grouped bar chart: mean vs median vs p90
metrics_data = []
for proc in TARGET_PROCS:
    sub = dff[dff['Procedure_Name'] == proc]
    total_gap = sub['gap_total'].dropna()
    metrics_data.append({
        'Procedure': proc.title(),
        'Median': total_gap.median(),
        'Mean': total_gap.mean(),
        'P90': total_gap.quantile(0.9),
    })

metrics_df = pd.DataFrame(metrics_data).set_index('Procedure')

x = np.arange(len(metrics_df))
width = 0.25

bars1 = ax.bar(x - width, metrics_df['Median'], width, label='Median',
               color=ACCENT_GREEN, edgecolor='white')
bars2 = ax.bar(x, metrics_df['Mean'], width, label='Mean',
               color=ACCENT_AMBER, edgecolor='white')
bars3 = ax.bar(x + width, metrics_df['P90'], width, label='P90 (worst 10%)',
               color=ACCENT_RED, edgecolor='white')

# Value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 2,
                f'{h:.0f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics_df.index, rotation=15, ha='right')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Total Gap: Actual − Planned (minutes)')
ax.set_title('Total Overrun: Median vs Mean vs P90', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('step_gap_p90.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Deep Dive: EKSPLORATIV LAPAROTOMI

Laparotomy is the most striking case: **+29 min mean prep gap but near-zero in-OR gap**. The surgical team runs on time once they start — the system is failing them upstream. Let's look at contributing factors.

In [ ]:
lap = dff[dff['Procedure_Name'] == 'EKSPLORATIV LAPAROTOMI'].copy()

# Acute vs planned breakdown
print('EKSPLORATIV LAPAROTOMI — Prep gap (actual − planned), by acute status:')
print()
for is_acute, label in [(False, 'Planned'), (True, 'Acute')]:
    sub = lap[lap['Is_Acute'] == is_acute]['gap_prep_to_inOR'].dropna()
    if len(sub) > 0:
        print(f'  {label}: n={len(sub)}, mean={sub.mean():+.1f}, '
              f'median={sub.median():+.1f}, p90={sub.quantile(0.9):+.1f}')

In [ ]:
# Day-of-week pattern for laparotomy prep gap
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

lap_dow = (
    lap
    .groupby('Day_of_Week')['gap_prep_to_inOR']
    .agg(['mean', 'count'])
    .reindex(day_order)
    .dropna()
)

fig, ax = plt.subplots(figsize=(10, 4))
colors = [ACCENT_GREEN if v <= 0 else ACCENT_RED if v == lap_dow['mean'].max()
          else RH_BLUE for v in lap_dow['mean']]

bars = ax.bar(lap_dow.index, lap_dow['mean'], color=colors, edgecolor='white')
for bar, n in zip(bars, lap_dow['count']):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'n={n:.0f}',
            ha='center', va='bottom', fontsize=9, color='#555')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean Prep Gap (min)')
ax.set_title('EKSPLORATIV LAPAROTOMI — Prep Step Overrun by Day of Week', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('laparotomi_prep_dow.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Room-level prep gap for laparotomy
MIN_CASES = 10

lap_room = (
    lap
    .groupby('Stue')['gap_prep_to_inOR']
    .agg(['mean', 'median', 'count'])
    .rename(columns={'mean': 'Mean Gap', 'count': 'Cases'})
    .query(f'Cases >= {MIN_CASES}')
    .sort_values('Mean Gap')
)

if len(lap_room) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(lap_room) * 0.5)))
    colors = [ACCENT_GREEN if v <= 0 else ACCENT_RED if v == lap_room['Mean Gap'].max()
              else RH_BLUE for v in lap_room['Mean Gap']]
    bars = ax.barh(lap_room.index, lap_room['Mean Gap'], color=colors, edgecolor='white')
    for bar, n in zip(bars, lap_room['Cases']):
        w = bar.get_width()
        ax.text(w + 1 if w >= 0 else w - 1, bar.get_y() + bar.get_height()/2,
                f'n={n}', va='center', ha='left' if w >= 0 else 'right',
                fontsize=9, color='#555')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Mean Prep Gap (actual − planned, min)')
    ax.set_title('EKSPLORATIV LAPAROTOMI — Prep Overrun by Room', fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('laparotomi_prep_room.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'No rooms with >= {MIN_CASES} cases. Try lowering MIN_CASES.')

---

## Deep Dive: JJ-KATETER — Highest Total Overrun

JJ catheter placement has the highest total mean gap (+39 min) with overrun in *both* steps. This suggests compound delays: late prep cascades into a rushed or delayed procedure.

In [ ]:
jj = dff[dff['Procedure_Name'] == 'JJ-KATETER ANLÆGGELSE'].copy()

# Is the in-OR gap correlated with the prep gap?
# i.e., does a late start make the procedure itself run longer?
corr_data = jj[['gap_prep_to_inOR', 'gap_inOR_to_leave']].dropna()

if len(corr_data) > 10:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(corr_data['gap_prep_to_inOR'], corr_data['gap_inOR_to_leave'],
              alpha=0.3, s=20, color=RH_BLUE, edgecolors='white', linewidth=0.3)
    
    # Trend line
    z = np.polyfit(corr_data['gap_prep_to_inOR'], corr_data['gap_inOR_to_leave'], 1)
    p = np.poly1d(z)
    x_range = np.linspace(corr_data['gap_prep_to_inOR'].min(),
                          corr_data['gap_prep_to_inOR'].max(), 100)
    ax.plot(x_range, p(x_range), color=ACCENT_RED, linewidth=2, linestyle='--')
    
    r = corr_data['gap_prep_to_inOR'].corr(corr_data['gap_inOR_to_leave'])
    ax.set_xlabel('Prep Step Gap (min)')
    ax.set_ylabel('In-OR Step Gap (min)')
    ax.set_title(f'JJ-KATETER: Do Late Preps Cause Longer Procedures?\n'
                 f'(r = {r:.2f}, n = {len(corr_data)})', fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax.axvline(0, color='black', linewidth=0.5, linestyle=':')
    plt.tight_layout()
    plt.savefig('jj_prep_vs_inor_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Not enough data for correlation analysis.')

In [ ]:
# JJ-Kateter: acute vs planned split
print('JJ-KATETER ANLÆGGELSE — Gap breakdown by acute status:')
print()
for is_acute, label in [(False, 'Planned'), (True, 'Acute')]:
    sub = jj[jj['Is_Acute'] == is_acute]
    prep = sub['gap_prep_to_inOR'].dropna()
    inor = sub['gap_inOR_to_leave'].dropna()
    total = sub['gap_total'].dropna()
    if len(prep) > 0:
        print(f'  {label} (n={len(prep)}):')
        print(f'    Prep gap:  mean={prep.mean():+.1f}, median={prep.median():+.1f}')
        print(f'    In-OR gap: mean={inor.mean():+.1f}, median={inor.median():+.1f}')
        print(f'    Total gap: mean={total.mean():+.1f}, median={total.median():+.1f}')
        print()

---

## Deep Dive: GASTROSKOPI — Volume Leader

Gastroscopy is high-volume and relatively short. Even a modest mean overrun of +30 minutes per case adds up at scale.

In [ ]:
gastro = dff[dff['Procedure_Name'] == 'GASTROSKOPI'].copy()

# Trend over time: is the gap improving or worsening?
gastro['Month'] = gastro['Dato'].dt.to_period('M')

monthly = (
    gastro
    .groupby('Month')
    .agg(
        prep_gap=('gap_prep_to_inOR', 'mean'),
        inor_gap=('gap_inOR_to_leave', 'mean'),
        total_gap=('gap_total', 'mean'),
        n=('gap_total', 'count'),
    )
)

fig, ax = plt.subplots(figsize=(12, 5))
months = monthly.index.astype(str)
ax.plot(months, monthly['prep_gap'], marker='o', color=ACCENT_AMBER,
        linewidth=2, label='Prep gap')
ax.plot(months, monthly['inor_gap'], marker='s', color=GREY,
        linewidth=2, label='In-OR gap')
ax.plot(months, monthly['total_gap'], marker='D', color=ACCENT_RED,
        linewidth=2.5, label='Total gap', linestyle='--')

ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.set_xlabel('Month')
ax.set_ylabel('Mean Gap (minutes)')
ax.set_title('GASTROSKOPI — Monthly Trend in Step Duration Gaps', fontweight='bold')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('gastro_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Annualized impact estimate for gastroskopi
n_cases = gastro['Case-ID Anonymous'].nunique()
mean_total_gap = gastro['gap_total'].mean()

print(f'GASTROSKOPI annualized impact estimate:')
print(f'  Unique cases in data: {n_cases}')
print(f'  Mean total overrun: {mean_total_gap:+.1f} min/case')
print(f'  Total excess time: {n_cases * mean_total_gap:,.0f} min = {n_cases * mean_total_gap / 60:,.0f} hours')
print(f'  If prep gap alone were eliminated: {n_cases * gastro["gap_prep_to_inOR"].mean():,.0f} min = '
      f'{n_cases * gastro["gap_prep_to_inOR"].mean() / 60:,.0f} hours saved')

---

## Summary

| Procedure | Prep Gap (mean) | In-OR Gap (mean) | Total Gap (mean) | P90 Total | Key Insight |
|---|---|---|---|---|---|
| Eksplorativ Laparotomi | **+29 min** | −1 min | +29 min | +142 min | Surgeons run on time — the system fails them before they start |
| JJ-Kateter Anlæggelse | **+30 min** | +9 min | +39 min | +90 min | Compound delay: late prep AND in-OR overrun |
| Gastroskopi | **+22 min** | +7 min | +30 min | — | High volume amplifies a moderate per-case gap |

### The common pattern

Across all three procedures, the **prep-to-patient-in-OR step** is the dominant source of time overrun. The procedures themselves run close to plan. This means the opportunity is in **logistics, not surgery**: room turnover, equipment staging, patient transport, and handoff protocols.

### Recommended actions

1. **Audit the prep step** for all three procedures: what specifically happens between planned prep start and actual patient arrival in OR? Where are the bottlenecks?
2. **Track the P90 cases**: the worst 10% of laparotomy cases overrun by 142+ minutes. These tail cases likely cascade delays to subsequent scheduled operations.
3. **Investigate JJ-Kateter's compound delay**: why does the in-OR step also overrun? Is it because a late start changes the team's pace, or are there clinical factors?

In [ ]:
print('Notebook complete. Charts saved:')
print('  - step_gap_stacked.png')
print('  - step_gap_distributions.png')
print('  - step_gap_p90.png')
print('  - laparotomi_prep_dow.png')
print('  - laparotomi_prep_room.png')
print('  - jj_prep_vs_inor_correlation.png')
print('  - gastro_monthly_trend.png')